In [1]:
import numpy as np
from nltk.corpus import gutenberg

text = gutenberg.raw('shakespeare-macbeth.txt')  
corpus = text.split()

vocab = list(set(corpus))
w2i = {w:i for i,w in enumerate(vocab)}
i2w = {i:w for i,w in enumerate(vocab)}

d = 10  # embedding dimension, your choice
E = np.random.randn(len(vocab), d)  # fresh, random embedding matrix

# fixed-window 

In [7]:
# et: Concatenated embeddings
# getting the embeddings for my words to feed into my nueral net
def build_window_input(words, w2i, E):
    indices = [w2i[w] for w in words]
    vectors = E[indices]
    e_t =vectors.flatten()
    return e_t

words = corpus[:5]
window = build_window_input(words, w2i, E)
print(window.shape)

# ht: hidden representation 
# building ony forward pass of my nn for rnn
np.random.seed(42)
# i want 16 neurons in my hidden layer
W = np.random.randn(16, len(window))
b_h = np.random.randn(16)

def hidden_layer(e_t, W, b_h):
    return np.tanh(W @ e_t + b_h)

# y^_hat: Next-token distribution 
from scipy.special import softmax
U = np.random.randn(len(vocab), 16)   # (|V|, hidden_size)
b_o = np.random.randn(len(vocab))     # (|V|,)


def output_layer(h_t, U, b_o):
    return softmax(U @ h_t +b_o)

(50,)


In [ ]:

# 1 FORWARD PASS
# Pass concatenated embeddings through hidden layer
h_t = hidden_layer(window, W, b_h)

# Pass hidden representation through output layer
y_hat = output_layer(h_t, U, b_o)

# checking the model shape
print("Input words:", words)

print("\nShapes:")
print("window e_t:", window.shape)
print("hidden h_t:", h_t.shape)
print("output y_hat:", y_hat.shape)

print("\nProbability check:")
print("Sum of probabilities:", y_hat.sum())

# look at top 5 probs after one forward pass
top_5_indices = np.argsort(y_hat)[-5:][::-1]
print("\nTop 5 predicted next words:")

for idx in top_5_indices:
    print(i2w[idx], y_hat[idx])

Input words: ['[The', 'Tragedie', 'of', 'Macbeth', 'by']

Shapes:
window e_t: (50,)
hidden h_t: (16,)
output y_hat: (5400,)

Probability check:
Sum of probabilities: 1.0

Top 5 predicted next words:
Macbeth. 0.2160488077703153
Cozen, 0.10389052621428443
Winde, 0.04488620406959136
Trees 0.03295388453658792
swarme 0.030430357147167453


# Vanilla RNN

In [10]:
np.random.seed(42)
# embedding dimension
d = 10

# hidden state size: number of neurons / values in the RNN memory
m = 16

# weights for the CURRENT word embedding x_t
W_x = np.random.randn(m, d)

# weights for the PREVIOUS hidden state h_(t-1)
W_h = np.random.randn(m ,m)

# bias for calculating the new hidden state
b_h = np.random.randn(m)

# initial hidden state: before the RNN has seen any words
h_0 = np.zeros(m)

def rnn_step(x_t, h_prev, W_x, W_h, b_h):
    h_t = np.tanh(W_h @ h_prev + W_x @ x_t + b_h)
    return h_t

h_prev = h_0

for word in corpus[:3]:
    # 1. get embedding for current word
    x_t = E[w2i[word]]

    # 2. calculate new hidden state
    h_t = rnn_step(x_t, h_prev, W_x, W_h, b_h)

    # 3. inspect it
    print("Word:", word)
    print("Hidden state shape:", h_t.shape)

    # 4. current hidden state becomes previous state
    #    for the NEXT word
    h_prev = h_t

Word: [The
Hidden state shape: (16,)
Word: Tragedie
Hidden state shape: (16,)
Word: of
Hidden state shape: (16,)
